# FACULTY SOLUTION — Week 2: Demand analysis and rack pricing
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['cluster_params', 'cluster_price_volume', 'pricing_params', 'fuel_nonfuel', 'worked_example_cluster', 'worked_example_params']}

In [ ]:

def compute(d):
    cp = d['cluster_params']; pv = d['cluster_price_volume']; pr = {x.parameter: x.value for x in d['pricing_params'].itertuples()}
    r = {}
    for c in cp.itertuples():
        s = pv[pv.cluster == c.cluster]
        r['est_' + c.cluster] = float(np.polyfit(s.ln_price, s.ln_volume, 1)[0])
        r['design_' + c.cluster] = float(c.design_elasticity)
    cor = cp[cp.market == 'Cordell']; eu = cp[cp.market == 'Europe']
    r['cordell_weighted_est'] = float(sum(r['est_' + x.cluster] * x.vol_share for x in cor.itertuples()))
    r['europe_weighted_est'] = float(sum(r['est_' + x.cluster] * x.vol_share for x in eu.itertuples()))
    r['cordell_weighted_passthrough'] = float((cor.passthrough * cor.vol_share).sum())
    for name in ['urban_high_comp', 'rural_low_comp']:
        pt = float(cp.loc[cp.cluster == name, 'passthrough'].iloc[0])
        r['vol_response_pct_' + name] = -r['est_' + name] * (pr['rack_cut'] * pt / pr['pump_base']) * 100
    return r


In [ ]:
fx = json.load(open('../fixtures/week2_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')